In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings("ignore")


In [17]:
#load data

df = pd.read_csv('MASTER SHEET - transformed_cases_2024.csv')
df.head()


,Province,District,moh_area,week,cases,avg_temperature_2m_mean,avg_temperature_2m_max,avg_temperature_2m_min,avg_precipitation_sum,avg_relative_humidity_2m_mean
0,Eastern Province,Kalmunai RDHS,MOH-Addalachchenai,1,5.0,25.886309,28.278570,23.964285,7.628572,87.357422
1,Eastern Province,Kalmunai RDHS,MOH-Addalachchenai,2,3.0,25.955952,28.042858,24.285715,16.328571,82.071373
2,Eastern Province,Kalmunai RDHS,MOH-Addalachchenai,3,8.0,25.662798,28.635714,22.821428,2.471429,83.079887
3,Eastern Province,Kalmunai RDHS,MOH-Addalachchenai,4,11.0,26.674702,29.464285,23.378571,0.742857,72.721458
4,Eastern Province,Kalmunai RDHS,MOH-Addalachchenai,5,4.0,26.046429,29.228573,23.542858,6.642857,86.899406


In [18]:
print('\n--- Data Types ---')
print(df.dtypes)
print('\n--- Basic Statistics ---')
df.describe()


--- Data Types ---
Province                          object
District                          object
moh_area                          object
week                               int64
cases                            float64
avg_temperature_2m_mean          float64
avg_temperature_2m_max           float64
avg_temperature_2m_min           float64
avg_precipitation_sum            float64
avg_relative_humidity_2m_mean    float64
dtype: object

--- Basic Statistics ---


,week,cases,avg_temperature_2m_mean,avg_temperature_2m_max,avg_temperature_2m_min,avg_precipitation_sum,avg_relative_humidity_2m_mean
count,38691.000000,38690.000000,38689.000000,38689.000000,38689.000000,38689.000000,38583.000000
mean,53.501383,2.611889,26.157421,30.180132,23.169247,7.245722,81.882345
std,30.599411,5.080387,2.317103,2.805926,2.568002,8.957673,7.421703
min,1.000000,0.000000,14.133440,15.826000,0.000000,0.000000,21.836020
25%,27.000000,0.000000,25.154170,28.707140,22.335710,1.742857,77.555920
50%,54.000000,1.000000,26.275890,29.935710,23.514290,4.728571,83.499200
75%,80.000000,3.000000,27.600595,31.935710,24.692858,9.371428,87.190180
max,107.000000,196.000000,36.871430,39.007140,79.728570,92.486590,98.732650


In [19]:
# Remove duplicate rows
df.duplicated().sum()
df.shape

(38691, 10)

In [20]:
# Visualise missing values
miss       = df.isnull().sum()
miss_pct   = (miss / len(df) * 100).round(3)
miss_df    = pd.DataFrame({'missing_count': miss, 'missing_%': miss_pct})
miss_df    = miss_df[miss_df['missing_count'] > 0]

print('Missing value summary:')
print(miss_df.to_string())



Missing value summary:
                               missing_count  missing_%
cases                                      1      0.003
avg_temperature_2m_mean                    2      0.005
avg_temperature_2m_max                     2      0.005
avg_temperature_2m_min                     2      0.005
avg_precipitation_sum                      2      0.005
avg_relative_humidity_2m_mean            108      0.279


In [21]:
#missing data is extremely minimal, even that is only 0.279%
# Calculate rows before
initial_rows = len(df)

# Drop any row that has a NaN in the Target or the specific Numeric Columns
cols_to_check = ['cases', 'avg_temperature_2m_mean', 'avg_temperature_2m_max', 
                 'avg_temperature_2m_min', 'avg_precipitation_sum', 
                 'avg_relative_humidity_2m_mean']

df.dropna(subset=cols_to_check, inplace=True)

# Summary of the cleanup
print(f"Cleaned! Removed {initial_rows - len(df)} total rows.")
print(f"Remaining rows: {len(df)}")

Cleaned! Removed 108 total rows.
Remaining rows: 38583


In [22]:
df.shape

(38583, 10)

In [23]:
TARGET_COL = "cases"
df[TARGET_COL] = df[TARGET_COL].astype(int)   # cases → integer
df['week']      = df['week'].astype(int)        # week  → integer
df['moh_area']  = df['moh_area'].str.strip()    # strip whitespace

print('Updated dtypes:')
print(df.dtypes)

Updated dtypes:
Province                          object
District                          object
moh_area                          object
week                               int64
cases                              int64
avg_temperature_2m_mean          float64
avg_temperature_2m_max           float64
avg_temperature_2m_min           float64
avg_precipitation_sum            float64
avg_relative_humidity_2m_mean    float64
dtype: object


In [24]:
# Cases: fill with 0 (no reported cases)
df["cases"] = df["cases"].fillna(0)

# Remove rows where max temp is logically impossible compared to min temp
df = df[df["avg_temperature_2m_max"] >= df["avg_temperature_2m_min"]]

# Combined Temperature Filter
df = df[
    (df["avg_temperature_2m_min"] > 10) & (df["avg_temperature_2m_min"] < 40) &
    (df["avg_temperature_2m_mean"] > 10) & (df["avg_temperature_2m_mean"] < 40) &
    (df["avg_temperature_2m_max"] < 45)
]

In [25]:
ALL_NUMERIC = [
    "avg_temperature_2m_mean",
    "avg_temperature_2m_max",
    "avg_temperature_2m_min",
    "avg_precipitation_sum",
    "avg_relative_humidity_2m_mean",
]
#Create a list of numeric columns WITHOUT the 'cases' column

for col in ALL_NUMERIC:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower=lower, upper=upper)

print(f' complete. Shape: {df.shape}')

 complete. Shape: (38575, 10)


In [26]:
#  Temperature range (diurnal variation — dengue vector sensitive)
df['temp_range'] = df['avg_temperature_2m_max'] - df['avg_temperature_2m_min']

#  Heat-Humidity Index
df['heat_humidity_index'] = (
    df['avg_temperature_2m_mean'] * df['avg_relative_humidity_2m_mean'] / 100
)

#  Cyclical encoding of week (captures seasonality)
df['week_sin'] = np.sin(2 * np.pi * df['week'] / 52)
df['week_cos'] = np.cos(2 * np.pi * df['week'] / 52)

#  Label-encode moh_area (for tree-based models)
le = LabelEncoder()
df['moh_area_encoded'] = le.fit_transform(df['moh_area'])

# Log1p of target (right-skewed count data)
df['cases_log1p'] = np.log1p(df[TARGET_COL])

print('New features created:')
new_cols = ['temp_range', 'heat_humidity_index', 'week_sin', 'week_cos',
            'moh_area_encoded', 'cases_log1p']
print(df[new_cols].describe().round(3))

New features created:
       temp_range  heat_humidity_index   week_sin   week_cos  \
count   38575.000            38575.000  38575.000  38575.000   
mean        6.902               21.410      0.003      0.018   
std         2.103                1.569      0.701      0.713   
min         2.000               13.562     -1.000     -1.000   
25%         5.307               20.439     -0.663     -0.663   
50%         6.679               21.775      0.000     -0.000   
75%         8.271               22.595      0.663      0.749   
max        16.136               24.928      1.000      1.000   

       moh_area_encoded  cases_log1p  
count         38575.000    38575.000  
mean            181.781        0.822  
std             105.403        0.869  
min               0.000        0.000  
25%              90.000        0.000  
50%             181.000        0.693  
75%             273.000        1.386  
max             364.000        5.283  


In [27]:
# avg_temperature_2m_min is highly correlated with avg_temperature_2m_mean so drop it
df = df.drop(columns=['avg_temperature_2m_min'])

In [28]:
scale_features = [
    'avg_temperature_2m_mean', 'avg_temperature_2m_max',
    'avg_precipitation_sum', 'avg_relative_humidity_2m_mean',
    'temp_range', 'heat_humidity_index', 'week_sin', 'week_cos'
]

ss = StandardScaler()
df[scale_features] = ss.fit_transform(df[scale_features])

print(df[scale_features].head(3).round(3).to_string())

   avg_temperature_2m_mean  avg_temperature_2m_max  avg_precipitation_sum  avg_relative_humidity_2m_mean  temp_range  heat_humidity_index  week_sin  week_cos
0                   -0.175                  -0.747                  0.206                          0.751      -1.230                0.767     0.167     1.366
1                   -0.140                  -0.837                  1.692                          0.017      -1.495               -0.069     0.337     1.336
2                   -0.286                  -0.611                 -0.675                          0.157      -0.517               -0.057     0.501     1.286


In [29]:
# Save cleaned dataset
df.to_csv("./data/Cleaned_Dengue_Data.csv", index=False)
print(f' Cleaned dataset saved ')

 Cleaned dataset saved 


In [30]:
df.shape

(38575, 15)